# Tarefa 2 · Importe total e número de vendas

Ficheiro `vendas_2025.csv`: uns **1,5 millóns de filas** (≈ 77 MB).

Na tarefa 1 liamos o ficheiro enteiro en memoria. Agora comparamos dúas opcións:

- **Opción A:** ler o ficheiro enteiro nun DataFrame.
- **Opción B:** ler por bloques (*chunks*), sen ter nunca o ficheiro completo en memoria.

As dúas deben dar o mesmo resultado.

In [ ]:
import time
from pathlib import Path
import pandas as pd

# Axusta a ruta se o CSV está noutro cartafol
FICHEIRO = Path("../datos/raw/vendas_2025-full.csv")

print(f"Tamaño do ficheiro: {FICHEIRO.stat().st_size / 1e6:.0f} MB")

## Opción A: ler o ficheiro enteiro en memoria

pandas non ten o límite de 1.048.576 filas dunha folla de cálculo: o único límite é a **memoria RAM** do equipo.

In [ ]:
inicio = time.perf_counter()

df = pd.read_csv(FICHEIRO, sep=";", decimal=",", encoding="utf-8")
assert pd.api.types.is_numeric_dtype(df["importe"]), "importe non se leu como número"

num_a, total_a = len(df), df["importe"].sum()
tempo_a = time.perf_counter() - inicio

print(f"Filas lidas: {num_a:,}".replace(",", "."))
print(f"Tempo: {tempo_a:.1f} s")
print(f"Memoria usada polo DataFrame: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

**Pregunta:** a memoria que ocupa o DataFrame é maior ou menor ca o tamaño do ficheiro en disco? Por que?

In [ ]:
df.head()

In [ ]:
# Liberamos a memoria antes da opción B
del df

## Opción B: ler por bloques (*chunks*)

Só ten en memoria **200.000 filas de cada vez**, e ademais só le a columna que necesitamos (`usecols`). Serviría tamén para ficheiros moito maiores ca a RAM.

In [ ]:
inicio = time.perf_counter()

num_b, total_b = 0, 0.0
for bloque in pd.read_csv(FICHEIRO, sep=";", decimal=",", encoding="utf-8",
                          usecols=["importe"], chunksize=200_000):
    num_b += len(bloque)
    total_b += bloque["importe"].sum()

tempo_b = time.perf_counter() - inicio
print(f"Filas lidas: {num_b:,}".replace(",", "."))
print(f"Tempo: {tempo_b:.1f} s")

## Comprobación: as dúas opcións deben coincidir

In [ ]:
assert num_a == num_b and round(total_a, 2) == round(total_b, 2)
print("As dúas opcións coinciden ✔")

## Resultado

In [ ]:
def eur(x):
    return f"{x:,.2f} €".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Número de vendas: {num_a:,}".replace(",", "."))
print(f"Importe total:    {eur(total_a)}")

## Para reflexionar

1. Que opción foi máis rápida no teu equipo? Por que cres que pasa?
2. Que pasaría coa opción A se o ficheiro ocupase 50 GB? E coa opción B?
3. Proba a cambiar o `chunksize` (por exemplo, 10.000 e 1.000.000). Como cambia o tempo?
4. Que vantaxe ten usar `usecols` na opción B? Probao tamén na opción A.